# Seed-robustness check (3 fine-tuning seeds)

Trains the classifier with three seeds and, for each, measures **comprehensiveness** of the four
explanation methods (+ a random baseline) on that seed's own stratified 60-instance sample.
Purpose: show that the method **ranking** (gradient/relevance > attention > random) is seed-stable.

**Before running (right panel):** Accelerator → **GPU T4/P100**; Internet → **ON**. Then Run All (~1.5–2 h).
Output to download from the **Output** tab: `seed_robustness.csv`.

In [ ]:
!pip -q install -U "transformers>=4.44" "datasets>=2.20" captum scikit-learn 2>/dev/null
import numpy as np, random, os, json, itertools, time, inspect
import torch
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding, set_seed)
from datasets import load_dataset
from sklearn.metrics import f1_score
print("torch", torch.__version__, "cuda", torch.cuda.is_available())

In [ ]:
# Config
SEEDS = [42, 123, 2024]
MODEL = "dbmdz/bert-base-turkish-128k-cased"
DATASET = "icgcihan/Turkish_Constutional_Court_Decisions"
TEXT, LABEL = "text", "labels"
MAX_LEN, EPOCHS, LR, BATCH, GRAD_ACC = 512, 3, 2e-5, 8, 2
BINS = [0.01, 0.05, 0.10, 0.20, 0.50]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(MODEL)
ds = load_dataset(DATASET)
def prep(b):
    e = tok(b[TEXT], truncation=True, max_length=MAX_LEN); e["labels"] = b[LABEL]; return e
keep = ds["train"].column_names
ds_tok = ds.map(prep, batched=True, remove_columns=keep)
collator = DataCollatorWithPadding(tok)

In [ ]:
# --- Explanation methods + comprehensiveness (embedded, GPU) ---
SPECIAL = set(tok.all_special_ids); MASK_ID = tok.mask_token_id
def softmax(x):
    import torch; return torch.softmax(x, -1)

def encode(text):
    e = tok(text, truncation=True, max_length=MAX_LEN, return_tensors="pt")
    return {k: v.to(DEVICE) for k, v in e.items()}

@torch.no_grad()
def prob_from_ids(model, ids, attn, target):
    return float(torch.softmax(model(input_ids=ids, attention_mask=attn).logits, -1)[0, target])

def content_positions(tokens, scores):
    idx = np.array([i for i, t in enumerate(tokens) if t not in ("[CLS]","[SEP]","[PAD]")])
    return idx[np.argsort(-scores[idx])]

def norm_scores(tokens, s):
    s = np.asarray(s, float)
    for i, t in enumerate(tokens):
        if t in ("[CLS]","[SEP]","[PAD]"): s[i] = 0.0
    s = np.clip(s, 0, None)
    return s / s.max() if s.max() > 0 else s

def attributions(model, text, rng):
    enc = encode(text); ids, attn = enc["input_ids"], enc["attention_mask"]
    out = model(**enc, output_attentions=True)
    probs = torch.softmax(out.logits, -1)[0]; pred = int(probs.argmax())
    tokens = tok.convert_ids_to_tokens(ids[0].tolist())
    res = {}
    # raw attention
    a = out.attentions[-1][0].mean(0)[0]
    res["raw_attention"] = norm_scores(tokens, a.detach().cpu().numpy())
    # rollout
    atts = [x[0].mean(0) for x in out.attentions]; seq = atts[0].size(0)
    eye = torch.eye(seq, device=DEVICE); R = eye.clone()
    for x in atts:
        xr = 0.5*x + 0.5*eye; xr = xr / xr.sum(-1, keepdim=True); R = xr @ R
    res["attention_rollout"] = norm_scores(tokens, R[0].detach().cpu().numpy())
    # chefer
    logit = out.logits[0, pred]
    grads = torch.autograd.grad(logit, out.attentions, retain_graph=False)
    Rc = torch.eye(seq, device=DEVICE)
    for A, G in zip(out.attentions, grads):
        cam = (G[0]*A[0]).clamp(min=0).mean(0); Rc = Rc + cam @ Rc
    res["chefer_relevance"] = norm_scores(tokens, Rc[0].detach().cpu().numpy())
    # IG
    from captum.attr import LayerIntegratedGradients
    ref = ids.clone()
    for i, t in enumerate(ids[0].tolist()):
        if t not in SPECIAL: ref[0, i] = tok.pad_token_id
    def fwd(x, m): return model(input_ids=x, attention_mask=m).logits
    lig = LayerIntegratedGradients(fwd, model.base_model.embeddings)
    at = lig.attribute(ids, baselines=ref, additional_forward_args=(attn,), target=pred,
                       n_steps=50, internal_batch_size=16).sum(-1).squeeze(0)
    res["integrated_gradients"] = norm_scores(tokens, at.detach().cpu().numpy())
    # random
    res["random"] = norm_scores(tokens, rng.random(len(tokens)))
    return tokens, pred, res

def comprehensiveness(model, text, tokens, pred, scores):
    enc = encode(text); ids, attn = enc["input_ids"], enc["attention_mask"]
    p0 = prob_from_ids(model, ids, attn, pred)
    ranked = content_positions(tokens, scores); n = len(ranked); vals = []
    for k in BINS:
        m = max(1, int(round(k*n)))
        c = ids.clone(); c[0, ranked[:m]] = MASK_ID
        vals.append(p0 - prob_from_ids(model, c, attn, pred))
    return float(np.mean(vals))

In [ ]:
# --- train + evaluate one seed ---
_ta = set(inspect.signature(TrainingArguments.__init__).parameters)
_ev = "eval_strategy" if "eval_strategy" in _ta else "evaluation_strategy"
def compute_metrics(p):
    import numpy as np
    return {"f1_macro": f1_score(p.label_ids, p.predictions.argmax(-1), average="macro")}

def run_seed(seed):
    set_seed(seed); random.seed(seed); np.random.seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2,
              attn_implementation="eager").to(DEVICE)
    kw = dict(output_dir=f"/kaggle/working/s{seed}", learning_rate=LR, num_train_epochs=EPOCHS,
              per_device_train_batch_size=BATCH, per_device_eval_batch_size=16,
              gradient_accumulation_steps=GRAD_ACC, weight_decay=0.01, warmup_ratio=0.1,
              logging_steps=100, save_strategy="no", seed=seed, fp16=torch.cuda.is_available(),
              report_to="none")
    kw[_ev] = "no"
    tr = Trainer(model=model, args=TrainingArguments(**kw), train_dataset=ds_tok["train"],
                 processing_class=tok, data_collator=collator, compute_metrics=compute_metrics)
    tr.train()
    # test predictions + confidence
    pr = tr.predict(ds_tok["test"]); probs = torch.softmax(torch.tensor(pr.predictions), -1).numpy()
    pred = probs.argmax(1); conf = probs.max(1); y = np.array(pr.label_ids)
    f1 = f1_score(y, pred, average="macro")
    import pandas as pd
    df = pd.DataFrame({"i": range(len(y)), "true": y, "pred": pred, "conf": conf, "correct": pred==y})
    corr = df[df.correct]; err = df[~df.correct]
    sel = pd.concat([corr.sort_values("conf", ascending=False).head(20),
                     corr.sort_values("conf").head(20),
                     err[err.pred==1].head(10), err[err.pred==0].head(10)]).drop_duplicates("i")
    rng = np.random.default_rng(seed)
    methods = ["raw_attention","attention_rollout","integrated_gradients","chefer_relevance","random"]
    acc = {m: [] for m in methods}
    model.eval()
    for _, r in sel.iterrows():
        text = str(ds["test"][int(r["i"])][TEXT])
        tokens, pd_, sc = attributions(model, text, rng)
        for m in methods:
            acc[m].append(comprehensiveness(model, text, tokens, pd_, sc[m]))
    row = {"seed": seed, "macro_f1": round(float(f1),3)}
    for m in methods: row[m] = round(float(np.mean(acc[m])),4)
    del model; torch.cuda.empty_cache()
    return row

In [ ]:
# --- run all seeds ---
import pandas as pd
rows = []
for s in SEEDS:
    t0 = time.time(); r = run_seed(s); rows.append(r)
    print(f"seed {s}: F1={r['macro_f1']} | comp:",
          {k: r[k] for k in ['integrated_gradients','chefer_relevance','raw_attention','attention_rollout','random']},
          f"({time.time()-t0:.0f}s)")
res = pd.DataFrame(rows)
res.to_csv("/kaggle/working/seed_robustness.csv", index=False)
print("\n=== Comprehensiveness by seed (higher = more faithful) ===")
print(res.to_string(index=False))
# ranking per seed
order = ["chefer_relevance","integrated_gradients","raw_attention","attention_rollout","random"]
print("\nRanking check (each seed, methods sorted by comprehensiveness):")
for _, r in res.iterrows():
    rk = sorted(order, key=lambda m: -r[m]); print(f"  seed {int(r['seed'])}: " + " > ".join(rk))

## Done
Download **`seed_robustness.csv`** from the Output tab and send it back. It contains, per seed, the
macro-F1 and the mean comprehensiveness of each method. If the ranking
(Integrated Gradients / Chefer > raw attention / rollout ≈ random) holds across all three seeds,
the single-seed limitation is closed.